# SenseVoice 数据量消融实验

验证不同数据量（25%/50%/75%）和两种数据增强策略对微调效果的影响。

In [ ]:
import torch
import torchaudio
import os
import json
import random
import numpy as np

# --- 固定随机种子，确保可复现 ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# --- 路径配置 ---
RAW_TRAIN = "/mnt/data/prepared_data/sensevoice/train.jsonl"
VAL_JSONL = "/mnt/data/prepared_data/sensevoice/val.jsonl"
PREPARED_DIR = "/mnt/data/prepared_data/sv_ablation"
os.makedirs(PREPARED_DIR, exist_ok=True)

# --- 读取原始数据 ---
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

raw_train = load_jsonl(RAW_TRAIN)
print(f"原始训练集: {len(raw_train)} 条")

# --- 子采样函数 ---
def subsample(data, ratio, output_path):
    """随机采样 ratio 比例的数据，保存为 jsonl"""
    if ratio <= 0 or ratio > 1:
        raise ValueError(f"ratio must be in (0, 1], got {ratio}")
    n = max(1, int(len(data) * ratio))
    sampled = random.sample(data, n)
    with open(output_path, 'w') as f:
        for item in sampled:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  子采样 {ratio:.0%}: {len(sampled)} 条 → {output_path}")
    return sampled

# --- 合成噪音增强 (A) ---
def add_gaussian_noise(audio_path, snr_db_range=(10, 20)):
    """加载音频，叠加高斯噪音，按随机 SNR"""
    wav, sr = torchaudio.load(audio_path)
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = torch.randn_like(wav) * torch.sqrt(noise_power)
    noisy_wav = wav + noise
    return noisy_wav, sr

def generate_noise_a_dataset(data, output_dir, output_filename="train_noise_a.jsonl"):
    """为每条数据生成一份带噪音版本，保存到 output_dir/noise/ 目录"""
    noise_dir = os.path.join(output_dir, "noise_a")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)

    new_data = []
    for item in data:
        src = item['source']
        try:
            noisy_wav, sr = add_gaussian_noise(src)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_a.wav")
            torchaudio.save(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")

    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  合成噪音增强: {len(new_data)} 条 → {jsonl_path}")
    return jsonl_path

# --- 真实噪音增强 (B) ---
DEMAND_DIR = "/mnt/data/demand"
DEMAND_NOISES = []
if os.path.exists(DEMAND_DIR):
    DEMAND_NOISES = sorted([
        os.path.join(DEMAND_DIR, f)
        for f in os.listdir(DEMAND_DIR)
        if f.endswith('.wav')
    ])
print(f"  真实噪音文件: {len(DEMAND_NOISES)} 条")

def add_real_noise(audio_path, noise_path, snr_db_range=(10, 20)):
    """叠加真实噪音"""
    wav, sr = torchaudio.load(audio_path)
    noise, noise_sr = torchaudio.load(noise_path)
    if noise_sr != sr:
        noise = torchaudio.functional.resample(noise, noise_sr, sr)
    if noise.shape[1] < wav.shape[1]:
        pad = torch.zeros(wav.shape[1] - noise.shape[1])
        noise = torch.cat([noise, pad.unsqueeze(0)], dim=1)
    else:
        noise = noise[:, :wav.shape[1]]
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = (noise ** 2).mean()
    if noise_power < 1e-10:
        noise_power = 1e-10
    adjusted_noise = noise * torch.sqrt(signal_power / (noise_power * (10 ** (snr_db / 10))))
    noisy_wav = wav + adjusted_noise
    return noisy_wav, sr

def generate_noise_b_dataset(data, output_dir, output_filename="train_noise_b.jsonl"):
    """使用 DEMAND 真实噪音增强"""
    if not DEMAND_NOISES:
        print("  警告: 未找到 DEMAND 噪音文件，跳过噪声B")
        return None
    noise_dir = os.path.join(output_dir, "noise_b")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)

    new_data = []
    for item in data:
        src = item['source']
        noise_path = random.choice(DEMAND_NOISES)
        try:
            noisy_wav, sr = add_real_noise(src, noise_path)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_b.wav")
            torchaudio.save(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")

    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  真实噪音增强: {len(new_data)} 条 → {jsonl_path}")
    return jsonl_path

In [ ]:
# --- 生成 5 个实验组的数据 ---
print("=" * 60)
print("开始生成实验数据...")
print("=" * 60)

# 实验组 1-3: 子采样
pct_configs = [
    (0.25, "/mnt/data/prepared_data/sv_ablation/train_25pct.jsonl"),
    (0.50, "/mnt/data/prepared_data/sv_ablation/train_50pct.jsonl"),
    (0.75, "/mnt/data/prepared_data/sv_ablation/train_75pct.jsonl"),
]

for ratio, out_path in pct_configs:
    subsample(raw_train, ratio, out_path)

# 实验组 4: 合成噪音增强 (A) → 200%
train_noise_a_path = generate_noise_a_dataset(
    raw_train,
    "/mnt/data/prepared_data/sv_ablation",
    "train_noise_a.jsonl"
)

# 实验组 5: 真实噪音增强 (B) → 200%
train_noise_b_path = generate_noise_b_dataset(
    raw_train,
    "/mnt/data/prepared_data/sv_ablation",
    "train_noise_b.jsonl"
)

print("\n数据生成完成!")
print(f"  25%:  {pct_configs[0][1]}")
print(f"  50%:  {pct_configs[1][1]}")
print(f"  75%:  {pct_configs[2][1]}")
print(f"  噪声A: {train_noise_a_path}")
print(f"  噪声B: {train_noise_b_path}")